# De Saxcé Contact: Full-Space Residual vs Reduced-Space Projection

This notebook compares two formulations of the De Saxcé bipotential contact law
on a 1-D porodynamics benchmark with slip-weakening friction at a fault boundary.

| Backend | State vector | Contact solve | Delassus operator $W$ |
|---------|-------------|---------------|-----------------------|
| **Full-space residual** (`build_dynamic_desaxce_residual_contact`) | $[y_{\mathrm{phys}},\, r]$ | Natural map rows in global Newton | Diagonal Schur approx in Jacobian |
| **Reduced-space projection** (`build_dynamic_desaxce_contact`) | $y_{\mathrm{phys}}$ only | Inner Newton with $W = U(A/h - J)^{-1}B$ | Full, exact |

Both solve the same contact problem — the same natural map
$\phi(r,u) = r_{\mathrm{eff}} - P_{K_\mu}(r_{\mathrm{eff}} - \rho\,\hat u)$ — and should
converge to the same trajectory. We compare:

1. **Solution accuracy** — $\|y^{\mathrm{full}} - y^{\mathrm{reduced}}\|_\infty$ at each time step
2. **Newton iteration counts** per step
3. **Wall-clock time**
4. **Convergence rate** (residual vs iteration within representative steps)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('')), ''))

import numpy as np
import matplotlib.pyplot as plt
import time

from skfem import Basis, ElementVector, ElementComposite, FacetBasis
from skfem.assembly import asm, BilinearForm, LinearForm
from skfem.helpers import grad, dot, ddot
from skfem import MeshLine, ElementLineP2, ElementLineP1
from scipy.sparse import lil_matrix, csr_matrix, block_diag as sp_block_diag
from scipy.sparse import vstack as sp_vstack

import solve_nivp
from solve_nivp.desaxce_contact import (
    build_dynamic_desaxce_contact,
    build_dynamic_desaxce_residual_contact,
)

## 1. Problem Setup — Poroelastic Beam with Fault Friction

A 1-D Biot poroelastic domain with a frictional fault at the right boundary.
Mixed $P_2/P_1$ elements for $(v, r, u, p)$ — solid velocity, relative fluid velocity,
displacement, and pressure — plus one accumulated-slip DOF.
Slip-weakening friction: $\mu(\delta) = \mu_{\mathrm{res}}\,(1 - \Delta\mu/\mu_{\mathrm{res}}\,e^{-\delta/D_c})$.

In [ ]:
# ── Material parameters ──
L = 1e-3  # length scale

rho_f = 1000 * L**3
rho_s = 2500 * L**3
Kf = 2.2e9 * L**1
Ks = 80e9 * L**1
nu = 0.15
phi = 0.2
alpha_biot = 0.6
turt = 1

k_vv = 1e18 * L**3
k_vu = 1e18 * L**3
k_r = 1e21 * L**3
eta_s = 9e-2

k_perm = 1e-18 * L**-2
mu_f = 1e-3 * L**1

Kb = Ks * (1 - alpha_biot)
G = 3/2 * Kb * (1 - 2*nu) / (1 + nu)
M_biot = ((alpha_biot - phi)/Ks + phi/Kf)**-1

rho_11 = (1 - phi) * rho_s + (turt - 1) * phi * rho_f
rho_22 = turt * phi * rho_f
rho_12 = -(turt - 1) * phi * rho_f
b_drag = (k_perm / mu_f)**-1
mu = G
lam = Kb - 2*G/3

robin_left = (k_vv, k_vv, k_vu, k_vu, k_r)
robin_right = (k_vv, 0, k_vu, 0, k_r)

# Fault friction parameters
Dc = 0.10 / 1000 / L
Dmu = -0.1
mu_res = 0.5
s11_eff_0 = 4e6 * L**1
D_domain = 1000.0

eps_slip = 1e-20 / L
slip_rate_floor = np.sqrt(eps_slip)

def mu_fric(slip, slip_rate):
    return mu_res * (1.0 - Dmu / mu_res * np.exp(-slip / Dc))

print(f"G = {G*1e-9*L**-1:.2f} GPa, Kb = {Kb*1e-9*L**-1:.2f} GPa")
print(f"s11_eff_0 = {s11_eff_0:.2e}, mu(0) = {mu_fric(0,0):.3f}")

In [ ]:
# ── Mesh and FEM ──
xmin, xmax = -D_domain/2, D_domain/2
xmin_d, xmax_d = xmin/L, xmax/L

mesh_elements = 41
xcoords = np.linspace(xmin_d, xmax_d, mesh_elements + 1)
mesh = MeshLine(xcoords)
mesh = mesh.with_boundaries({
    'left': lambda x: np.isclose(x[0], xmin_d),
    'right': lambda x: np.isclose(x[0], xmax_d),
})

dim = 2
intorder = 4
el_v = ElementVector(ElementLineP2(), dim=dim)
el_r = ElementVector(ElementLineP2(), dim=dim)
el_u = ElementVector(ElementLineP2(), dim=dim)
el_p = ElementLineP1()
mixed_element = ElementComposite(el_v, el_r, el_u, el_p)

basis = Basis(mesh, mixed_element, intorder=intorder)
fbasis_left = FacetBasis(mesh, mixed_element, intorder=intorder,
                         facets=mesh.boundaries['left'])
fbasis_right = FacetBasis(mesh, mixed_element, intorder=intorder,
                          facets=mesh.boundaries['right'])

print(f"Physical DOFs: {basis.N}, mesh elements: {mesh_elements}")

In [ ]:
# ── Variational forms ──
def sij_core(E_kl):
    dim_t = E_kl.shape[-1]
    batch_shape = E_kl.shape[:-2]
    if dim_t == 2:
        E3D = np.zeros(batch_shape + (3, 3), dtype=E_kl.dtype)
        E3D[..., :dim_t, :dim_t] = E_kl
    else:
        E3D = E_kl
    trE = np.trace(E3D, axis1=-2, axis2=-1)[..., None, None]
    I = np.eye(3, dtype=E3D.dtype)
    S3 = 2.0 * mu * E3D + lam * trE * I
    return S3[..., :dim_t, :dim_t]

def Sij(E):
    E_last = np.moveaxis(np.asarray(E), (0, 1), (-2, -1))
    return np.moveaxis(sij_core(E_last), (-2, -1), (0, 1))

def make_sym(gradu):
    o = np.zeros((2, 2, *gradu.shape[2:]), dtype=gradu.dtype)
    o[0, 0] = gradu[0, 0]
    o[0, 1] = 0.5 * gradu[1, 0]
    o[1, 0] = 0.5 * gradu[1, 0]
    return o

def my_sym_grad(u):
    return make_sym(grad(u))

@BilinearForm
def varform_lhs(dv, dr, du, dp, dv_, dr_, du_, dp_, w):
    return (rho_11 * dot(dv, dv_) + rho_22 * dot(dr, dr_)
            + rho_12 * dot(dv, dr_) + rho_12 * dot(dr, dv_)
            + dot(du, du_) + dp * dp_)

@BilinearForm
def varform_rhs(v, r, u, p, dv_, dr_, du_, dp_, w):
    Eps = my_sym_grad(u)
    Su = Sij(Eps)
    dEps = my_sym_grad(v)
    Sv = eta_s * Sij(dEps)
    I = np.eye(Eps.shape[0])[:, :, None, None] * np.ones_like(Eps)
    S = Su + Sv - alpha_biot * I * p
    term1 = ddot(S, my_sym_grad(dv_)) + b_drag * dot(r, dr_) - dot(v, du_)
    term2 = -dot(p, grad(dr_)[0, 0]) + M_biot * dot(
        alpha_biot * grad(v)[0, 0] + grad(r)[0, 0], dp_)
    return term1 + term2

@BilinearForm
def varform_robin(v, r, u, p, dv_, dr_, du_, dp_, w):
    c_vv1, c_vv2, c_vu1, c_vu2, c_r = w.params
    return ((c_vv1 * v[0] + c_vu1 * u[0]) * dv_[0]
            + (c_vv2 * v[1] + c_vu2 * u[1]) * dv_[1]
            + c_r * r[0] * dr_[0])

E_mass = asm(varform_lhs, basis)
A_bulk = -asm(varform_rhs, basis)
A_bndr_L = -asm(varform_robin, fbasis_left, params=robin_left)
A_bndr_R = -asm(varform_robin, fbasis_right, params=robin_right)
A_stiff = A_bulk + A_bndr_L + A_bndr_R

In [ ]:
# ── Dirichlet BCs (symmetric elimination) ──
user_to_skfem = {
    "v1": "u^1^1", "v2": "u^2^1",
    "r1": "u^1^2", "r2": "u^2^2",
    "u1": "u^1^3", "u2": "u^2^3",
    "p": "u^4",
}
bc_dofs_spec = {'u1': {'left': 0.0}, 'u2': {'left': 0.0}}

E_D = E_mass.tolil()
A_D = A_stiff.tolil()
dofs_D = np.array([], dtype=np.int32)
for field, sides_bc in bc_dofs_spec.items():
    for side, _ in sides_bc.items():
        dofs_D = np.append(dofs_D,
                           basis.get_dofs(side).all(user_to_skfem[field]))
E_D[dofs_D, :] = 0.; E_D[:, dofs_D] = 0.; E_D[dofs_D, dofs_D] = 1.
A_D[dofs_D, :] = 0.; A_D[:, dofs_D] = 0.
E_D = E_D.tocsr(); A_D = A_D.tocsr()

ndofs = basis.N

# Augment with accumulated slip DOF
slip_idx = ndofs
E_D = sp_block_diag([E_D, csr_matrix(np.array([[1.]]))], format='csr')
A_D = sp_block_diag([A_D, csr_matrix(np.array([[0.]]))], format='csr')
ndofs_aug = ndofs + 1

print(f"Augmented system: {ndofs} physics + 1 slip = {ndofs_aug} DOFs")

In [ ]:
# ── Contact frame and extraction operators ──
vtn_idx = basis.get_dofs('right').all(user_to_skfem['v1'])[0]
vtt_idx = basis.get_dofs('right').all(user_to_skfem['v2'])[0]
u1n_idx = basis.get_dofs('right').all(user_to_skfem['u1'])[0]
u2t_idx = basis.get_dofs('right').all(user_to_skfem['u2'])[0]

contact_vel_C = lil_matrix((2, ndofs_aug), dtype=float)
contact_vel_C[0, vtn_idx] = 1.0
contact_vel_C[1, vtt_idx] = 1.0
contact_vel_C = contact_vel_C.tocsr()

contact_gap_C = lil_matrix((2, ndofs_aug), dtype=float)
contact_gap_C[0, u1n_idx] = -1.0
contact_gap_C[1, u2t_idx] = 1.0
contact_gap_C = contact_gap_C.tocsr()

contact_B = contact_vel_C.T.tocsr()

contact_vel_idx = np.array([vtn_idx, vtt_idx], dtype=int)
contact_other_idx = np.array(
    sorted(set(range(ndofs_aug)) - set(contact_vel_idx.tolist())), dtype=int)
contact_component_slices = [contact_vel_idx, contact_other_idx]

In [ ]:
# ── RHS, Jacobian, and friction law ──
_jac_placeholder = 1e-300
A_D_jac_tpl = A_D.tolil()
A_D_jac_tpl[slip_idx, 0] = _jac_placeholder
A_D_jac_tpl[slip_idx, vtt_idx] = _jac_placeholder
A_D_jac_tpl = A_D_jac_tpl.tocsr()

def rhs_plant(t, y):
    out = A_D @ y
    v2 = y[vtt_idx]
    out[slip_idx] = np.sqrt(v2**2 + eps_slip) - slip_rate_floor
    return out

def rhs_jac_plant(t, y):
    J = A_D_jac_tpl.copy()
    v2 = y[vtt_idx]
    J[slip_idx, vtt_idx] = v2 / np.sqrt(v2**2 + eps_slip)
    return J

def mu_(z):
    return mu_fric(z[slip_idx], z[vtt_idx])

contact_blocks = [dict(vel_normal_idx=0, vel_tangential_idx=[1], mu=mu_)]

def contact_s0(y):
    return np.array([s11_eff_0])

def contact_w0(y, k):
    return np.array([-s11_eff_0 * mu_fric(0.0, 0.0) * 1.001])

In [ ]:
# ── Initial condition ──
def IC(xn):
    x = xn[0] * L
    return 1e-6 * (x + D_domain / 2) / D_domain / L

y0 = np.zeros(ndofs)
(vi, vb), (ri, rb), (ui, ub), (pi, pb) = basis.split(y0)
(v1i, vb1), (v2i, vb2) = vb.split(y0)
idx2 = vb.split_indices()[1]
vi[idx2] = vb1.project(IC)
y0[basis.split_indices()[0]] = vi
y0 = np.append(y0, 0.0)  # accumulated slip

## 2. Build Both Backends

Both backends receive identical physics, contact parameters, and prestress.

In [ ]:
# ── Backend 1: Full-space residual [y, r] ──
cs_full = build_dynamic_desaxce_residual_contact(
    A=E_D, rhs_smooth=rhs_plant, y0=y0,
    contacts=contact_blocks,
    gap_extract=contact_gap_C, vel_extract=contact_vel_C, B=contact_B,
    component_slices=contact_component_slices,
    contact_rho='auto', reaction_units='force',
    get_s0=contact_s0, get_w0=contact_w0,
    rhs_jac=rhs_jac_plant,
    inactive_handling='natural_map',
)

n_full = cs_full.y0.size
print(f"Full-space state: {n_full} DOFs ({ndofs_aug} phys + {n_full - ndofs_aug} react)")

In [ ]:
# ── Backend 2: Reduced-space projection ──
cs_red = build_dynamic_desaxce_contact(
    A=E_D, rhs_smooth=rhs_plant, y0=y0,
    contacts=contact_blocks,
    gap_extract=contact_gap_C, vel_extract=contact_vel_C, B=contact_B,
    component_slices=contact_component_slices,
    get_s0=contact_s0, get_w0=contact_w0,
    rhs_jac=rhs_jac_plant,
    inactive_handling='cone',
    smooth_rhs_is_affine=False,
)

n_red = cs_red.y0.size
print(f"Reduced-space state: {n_red} DOFs (physical only)")

## 3. Solver Configuration

Both use Backward Euler, PETSc linear solver, fixed step size.

In [ ]:
tmax = 5.0
n_steps = 100
h0 = tmax / n_steps
t_span = (0.0, tmax)

def make_solve_kwargs(cs, label):
    n_state = cs.y0.size
    nl_atol = np.full(n_state, 1e-8)
    nl_rtol = np.full(n_state, 1e-6)
    if label == 'full':
        nl_atol[ndofs_aug:] = 1e-10
        nl_rtol[ndofs_aug:] = 0.0

    solver_opts = dict(cs.solver_opts) if cs.solver_opts else {}
    solver_opts.update(
        tol=1e-6, max_iter=500,
        rhs_jac=cs.rhs_jac,
        linear_solver='petsc',
        globalization='linesearch',
    )

    integrator_opts = dict(cs.integrator_opts) if cs.integrator_opts else {}

    return dict(
        fun=cs.rhs, t_span=t_span, y0=cs.y0, A=cs.A,
        method='backward_euler',
        projection=cs.projection,
        solver='semismooth_newton',
        solver_opts=solver_opts,
        nl_atol=nl_atol, nl_rtol=nl_rtol,
        component_slices=cs.component_slices,
        integrator_opts=integrator_opts,
        h0=h0, adaptive=False,
        dae_var_weight='auto',
        active_set_filter=False,
        verbose=False,
    )

kw_full = make_solve_kwargs(cs_full, 'full')
kw_red = make_solve_kwargs(cs_red, 'reduced')

print(f"Time span: {t_span}, h = {h0:.4g}, steps = {n_steps}")
print(f"Linear solver: PETSc")

## 4. Run Both Backends

In [ ]:
print("Running full-space residual backend...")
t0 = time.perf_counter()
t_full, y_full, h_full, fk_full, info_full = solve_nivp.solve_ivp_ns(**kw_full)
wall_full = time.perf_counter() - t0
print(f"  Done: {len(t_full)-1} steps, {wall_full:.2f} s")

print("\nRunning reduced-space projection backend...")
t0 = time.perf_counter()
t_red, y_red, h_red, fk_red, info_red = solve_nivp.solve_ivp_ns(**kw_red)
wall_red = time.perf_counter() - t0
print(f"  Done: {len(t_red)-1} steps, {wall_red:.2f} s")

## 5. Solution Comparison

In [ ]:
# Compare physical DOFs (both backends share the first ndofs_aug components)
n_compare = min(len(t_full), len(t_red))
t_common = t_full[:n_compare]

diff_linf = np.array([
    np.max(np.abs(y_full[i, :ndofs_aug] - y_red[i, :ndofs_aug]))
    for i in range(n_compare)
])
diff_v2 = np.array([
    abs(y_full[i, vtt_idx] - y_red[i, vtt_idx])
    for i in range(n_compare)
])
diff_slip = np.array([
    abs(y_full[i, slip_idx] - y_red[i, slip_idx])
    for i in range(n_compare)
])

print(f"Max L-inf difference (physical state): {diff_linf.max():.3e}")
print(f"Max |v2| difference:                   {diff_v2.max():.3e}")
print(f"Max |slip| difference:                 {diff_slip.max():.3e}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Tangential velocity at fault
ax = axes[0, 0]
ax.plot(t_full, [y[vtt_idx] for y in y_full], '-', label='Full-space', lw=1.5)
ax.plot(t_red, [y[vtt_idx] for y in y_red], '--', label='Reduced-space', lw=1.5)
ax.set_xlabel('t'); ax.set_ylabel('$v_2$ (fault)')
ax.set_title('Tangential velocity at fault'); ax.legend()

# Accumulated slip
ax = axes[0, 1]
ax.plot(t_full, [y[slip_idx] for y in y_full], '-', label='Full-space', lw=1.5)
ax.plot(t_red, [y[slip_idx] for y in y_red], '--', label='Reduced-space', lw=1.5)
ax.set_xlabel('t'); ax.set_ylabel('Accumulated slip $\\delta$')
ax.set_title('Accumulated slip'); ax.legend()

# Solution difference
ax = axes[1, 0]
ax.semilogy(t_common[1:], diff_linf[1:], '-k', lw=1.5)
ax.set_xlabel('t'); ax.set_ylabel('$\\|y^{full} - y^{red}\\|_\\infty$')
ax.set_title('Solution difference')

# Normal velocity at fault
ax = axes[1, 1]
ax.plot(t_full, [y[vtn_idx] for y in y_full], '-', label='Full-space', lw=1.5)
ax.plot(t_red, [y[vtn_idx] for y in y_red], '--', label='Reduced-space', lw=1.5)
ax.set_xlabel('t'); ax.set_ylabel('$v_1$ (fault)')
ax.set_title('Normal velocity at fault'); ax.legend()

plt.tight_layout()
plt.show()

## 6. Newton Iteration Counts

In [ ]:
def extract_iters(info):
    """Extract iteration counts from solve_ivp_ns info list."""
    if not info or not isinstance(info, (list, tuple)):
        return np.array([])
    iters = []
    for entry in info:
        if isinstance(entry, tuple) and len(entry) >= 3:
            iters.append(int(entry[2]))
        elif isinstance(entry, dict) and 'iterations' in entry:
            iters.append(int(entry['iterations']))
    return np.array(iters, dtype=int)

iters_full = extract_iters(info_full)
iters_red = extract_iters(info_red)

print(f"{'':>25s} {'Full-space':>12s} {'Reduced-space':>14s}")
print(f"{'─'*55}")
if iters_full.size > 0:
    print(f"{'Mean iters/step':>25s} {iters_full.mean():12.1f} {iters_red.mean() if iters_red.size else 0:14.1f}")
    print(f"{'Max iters':>25s} {iters_full.max():12d} {iters_red.max() if iters_red.size else 0:14d}")
    print(f"{'Total iters':>25s} {iters_full.sum():12d} {iters_red.sum() if iters_red.size else 0:14d}")
print(f"{'Wall-clock (s)':>25s} {wall_full:12.2f} {wall_red:14.2f}")
print(f"{'Speedup':>25s} {'':>12s} {wall_full/wall_red if wall_red > 0 else 0:14.2f}x")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
if iters_full.size > 0:
    ax.bar(np.arange(len(iters_full)) - 0.2, iters_full, 0.4,
           label='Full-space', alpha=0.8)
if iters_red.size > 0:
    ax.bar(np.arange(len(iters_red)) + 0.2, iters_red, 0.4,
           label='Reduced-space', alpha=0.8)
ax.set_xlabel('Step'); ax.set_ylabel('Newton iterations')
ax.set_title('Newton iterations per time step')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Convergence Rate (Per-Step Residual History)

To assess convergence rate, we re-run a single representative step
and record the Newton residual at each iteration.

In [ ]:
from solve_nivp import ODESystem, ODESolver
from solve_nivp.integrations import BackwardEuler
from solve_nivp.nonlinear_solvers import ImplicitEquationSolver

def single_step_residual_history(cs, y_prev, t_prev, h_step, label):
    """Run one Backward Euler step and return per-iteration residual norms."""
    n_state = cs.y0.size
    nl_atol = np.full(n_state, 1e-8)
    nl_rtol = np.full(n_state, 1e-6)
    if label == 'full':
        nl_atol[ndofs_aug:] = 1e-10
        nl_rtol[ndofs_aug:] = 0.0

    solver = ImplicitEquationSolver(
        method='semismooth_newton',
        tol=1e-6, max_iter=500,
        nl_atol=nl_atol, nl_rtol=nl_rtol,
        rhs_jac=cs.rhs_jac,
        linear_solver='petsc',
        globalization='linesearch',
        record_residuals=True,
    )

    integrator = BackwardEuler(
        pass_prev_state=True, pass_step_size=True,
    )

    system = ODESystem(
        fun=cs.rhs, A=cs.A, method=integrator,
        projection=cs.projection, solver=solver,
        component_slices=cs.component_slices,
    )

    y_prev = np.asarray(y_prev, dtype=float)
    fk = cs.rhs(t_prev, y_prev)
    y_new, converged, n_iter, fk_new = system.step(
        t_prev, y_prev, h_step, fk)

    residuals = getattr(solver, '_residual_history', None)
    return residuals, converged, n_iter

# Pick a step where contact is active (after some slip has occurred)
test_step = min(20, len(t_full) - 2)
t_prev_test = t_full[test_step]
h_test = t_full[test_step + 1] - t_full[test_step]

res_full, conv_f, nit_f = single_step_residual_history(
    cs_full, y_full[test_step], t_prev_test, h_test, 'full')
res_red, conv_r, nit_r = single_step_residual_history(
    cs_red, y_red[test_step], t_prev_test, h_test, 'reduced')

print(f"Step {test_step} (t={t_prev_test:.4f}):")
print(f"  Full-space:    {nit_f} iters, converged={conv_f}")
print(f"  Reduced-space: {nit_r} iters, converged={conv_r}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

if res_full is not None and len(res_full) > 0:
    ax.semilogy(range(len(res_full)), res_full, 'o-',
                label=f'Full-space ({nit_f} iters)', lw=1.5, ms=5)
if res_red is not None and len(res_red) > 0:
    ax.semilogy(range(len(res_red)), res_red, 's--',
                label=f'Reduced-space ({nit_r} iters)', lw=1.5, ms=5)

ax.set_xlabel('Newton iteration')
ax.set_ylabel('Residual norm $\\|F\\|_\\infty$')
ax.set_title(f'Convergence rate at step {test_step} (t = {t_prev_test:.4f})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Summary

| Metric | Full-space $[y, r]$ | Reduced-space (projection) |
|--------|--------------------|--------------------------|
| State size | $n_{\mathrm{phys}} + n_{\mathrm{react}}$ | $n_{\mathrm{phys}}$ only |
| Delassus $W$ | Diagonal Schur approx | Full $U(A/h-J)^{-1}B$ |
| Jacobian consistency | Quasi-Newton (FD mismatch) | Exact in reduced space |
| Newton convergence | Superlinear | Quadratic (inner) |

Both formulations solve the same De Saxcé natural map and should converge to
identical physical trajectories up to solver tolerances.